# Fase 5 — Entrenamiento de la U-Net base (requiere GPU)

**Experimento base:** U-Net (32 canales, profundidad 4, con saltos) · solo los **252 chips dorados** de train · pérdida **BCE**.

**Plan de experimentos (para que veas a dónde va esto):**
| Fase | Experimento | Cambia respecto a la base |
|---|---|---|
| 5 | `base_bce` | — |
| 6 (b) | `dice`, `combinada` | función de pérdida |
| 6 (b) | `sin_saltos` o `profundidad_3` | arquitectura |
| 6 (a) | `mano+debil` | añade los 4,384 chips débiles (con la mejor pérdida de (b)) |
| 7 | evaluación final en **test** | solo el modelo elegido |

## Teoría de las decisiones de entrenamiento
- **AdamW** (Loshchilov & Hutter): Adam con *weight decay* desacoplado → regularización L2 que sí funciona con tasas adaptativas.
- **Calentamiento + coseno:** la tasa sube linealmente 3 épocas (al inicio los pesos son aleatorios y un paso grande desestabiliza la BatchNorm) y luego baja suavemente hasta ~0 → pasos finos al final.
- **Selección por IoU de validación:** guardamos el modelo de la época con mejor IoU en *valid*. **Test no se toca** hasta la fase 7: si eligiéramos con test, el número reportado sería optimista (fuga de información).
- **Parada temprana:** si en 20 épocas el IoU de valid no mejora, se detiene (ahorra GPU y evita sobreajuste).
- **Checkpoint por época en Drive:** si Colab se desconecta, al volver a correr la celda de entrenamiento **reanuda** donde quedó.

## 0. Preparación — verificar GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
RAIZ_PROYECTO = Path('/content/drive/MyDrive/rnap_p2_inundaciones')
sys.path.insert(0, str(RAIZ_PROYECTO))

!pip install -q rasterio
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
/bin/bash: line 1: nvidia-smi: command not found


## 1. Módulos nuevos

In [2]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/experimento.py
"""
experimento.py
Configuración de un experimento. Cada corrida (base y ablaciones) se describe con UNA instancia,
se guarda como JSON junto a sus resultados -> trazabilidad total para el informe.
"""
import json
from dataclasses import dataclass, asdict, field


@dataclass
class ConfiguracionExperimento:
    nombre: str = "base_bce"
    # --- datos ---
    fuente_entrenamiento: str = "mano"      # "mano" | "mano+debil" (fase 6)
    tamano_lote: int = 8
    tamano_recorte: int = 256
    # --- modelo ---
    canales_base: int = 32
    profundidad: int = 4
    usar_saltos: bool = True
    abandono: float = 0.0
    # --- pérdida ---
    perdida: str = "bce"                     # "bce" | "dice" | "combinada"
    parametros_perdida: dict = field(default_factory=dict)
    # --- optimización ---
    tasa_aprendizaje: float = 1e-3
    decaimiento_pesos: float = 1e-4          # regularización L2 desacoplada (AdamW)
    epocas: int = 100
    epocas_calentamiento: int = 3            # la tasa sube linealmente al inicio (evita saltos bruscos)
    paciencia: int = 20                      # parada temprana: épocas sin mejorar IoU de validación
    umbral: float = 0.5
    semilla: int = 42

    def guardar(self, ruta):
        with open(ruta, "w") as f:
            json.dump(asdict(self), f, indent=2, ensure_ascii=False)

    @classmethod
    def cargar(cls, ruta):
        with open(ruta) as f:
            return cls(**json.load(f))


Writing /content/drive/MyDrive/rnap_p2_inundaciones/src/experimento.py


In [3]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/evaluacion.py
"""
evaluacion.py
Evaluación de un modelo sobre un DataLoader completo (chips enteros de 512x512).
"""
import torch

from .metricas_torch import AcumuladorConfusion


@torch.no_grad()
def evaluar(modelo, cargador, dispositivo, perdida=None, umbral: float = 0.5):
    """Devuelve (acumulador con métricas globales y por evento, pérdida media o None)."""
    modelo.eval()
    acumulador = AcumuladorConfusion(umbral)
    suma_perdida, n_lotes = 0.0, 0
    usar_amp = dispositivo.type == "cuda"
    for imagenes, etiquetas, ids in cargador:
        imagenes = imagenes.to(dispositivo, non_blocking=True)
        etiquetas = etiquetas.to(dispositivo, non_blocking=True)
        with torch.autocast(dispositivo.type, dtype=torch.float16, enabled=usar_amp):
            logits = modelo(imagenes)
        acumulador.actualizar(logits, etiquetas, ids)
        if perdida is not None:
            suma_perdida += perdida(logits.float(), etiquetas).item()
            n_lotes += 1
    return acumulador, (suma_perdida / n_lotes if n_lotes else None)


@torch.no_grad()
def predecir_probabilidades(modelo, imagen, dispositivo):
    """imagen: tensor (2, H, W) -> mapa de probabilidad de agua (H, W) en CPU."""
    modelo.eval()
    logits = modelo(imagen[None].to(dispositivo))
    return torch.sigmoid(logits.float())[0, 0].cpu()


Writing /content/drive/MyDrive/rnap_p2_inundaciones/src/evaluacion.py


In [4]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/entrenamiento.py
"""
entrenamiento.py
Bucle de entrenamiento con:
  - AdamW + calentamiento lineal + decaimiento coseno de la tasa de aprendizaje
  - precisión mixta (AMP) en GPU
  - selección del mejor modelo por IoU de VALIDACIÓN (test no se mira hasta el final)
  - parada temprana
  - checkpoint en Drive en cada época -> si Colab se desconecta, se REANUDA donde quedó
"""
import json
import math
import time

import pandas as pd
import torch

from .evaluacion import evaluar
from .modelo_unet import UNet
from .perdidas import crear_perdida
from .utilidades import fijar_semilla


def crear_modelo(configuracion):
    return UNet(canales_entrada=2, canales_base=configuracion.canales_base,
                profundidad=configuracion.profundidad, usar_saltos=configuracion.usar_saltos,
                abandono=configuracion.abandono)


def _factor_tasa(epoca, configuracion):
    """Multiplicador de la tasa de aprendizaje en cada época (0-indexada)."""
    if epoca < configuracion.epocas_calentamiento:
        return (epoca + 1) / configuracion.epocas_calentamiento
    progreso = (epoca - configuracion.epocas_calentamiento) / max(1, configuracion.epocas - configuracion.epocas_calentamiento)
    return 0.5 * (1 + math.cos(math.pi * progreso))


def entrenar(configuracion, cargador_entrenamiento, cargador_validacion, dispositivo, dir_salida,
             reanudar: bool = True):
    dir_salida.mkdir(parents=True, exist_ok=True)
    configuracion.guardar(dir_salida / "configuracion.json")
    fijar_semilla(configuracion.semilla)

    modelo = crear_modelo(configuracion).to(dispositivo)
    perdida = crear_perdida(configuracion.perdida, **configuracion.parametros_perdida)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=configuracion.tasa_aprendizaje,
                                    weight_decay=configuracion.decaimiento_pesos)
    planificador = torch.optim.lr_scheduler.LambdaLR(optimizador, lambda e: _factor_tasa(e, configuracion))
    usar_amp = dispositivo.type == "cuda"
    escalador = torch.amp.GradScaler("cuda", enabled=usar_amp)

    epoca_inicial, mejor_iou, epocas_sin_mejora, historial = 0, -1.0, 0, []
    ruta_ultimo, ruta_mejor = dir_salida / "ultimo.pt", dir_salida / "mejor.pt"
    if reanudar and ruta_ultimo.exists():
        estado = torch.load(ruta_ultimo, map_location=dispositivo, weights_only=False)
        modelo.load_state_dict(estado["modelo"]); optimizador.load_state_dict(estado["optimizador"])
        planificador.load_state_dict(estado["planificador"]); escalador.load_state_dict(estado["escalador"])
        epoca_inicial, mejor_iou = estado["epoca"] + 1, estado["mejor_iou"]
        epocas_sin_mejora, historial = estado["epocas_sin_mejora"], estado["historial"]
        print(f"↻ Reanudando desde la época {epoca_inicial} (mejor IoU valid = {mejor_iou:.4f})")

    for epoca in range(epoca_inicial, configuracion.epocas):
        inicio = time.time()
        modelo.train()
        suma_perdida, n_lotes = 0.0, 0
        for imagenes, etiquetas, _ in cargador_entrenamiento:
            imagenes = imagenes.to(dispositivo, non_blocking=True)
            etiquetas = etiquetas.to(dispositivo, non_blocking=True)
            with torch.autocast(dispositivo.type, dtype=torch.float16, enabled=usar_amp):
                logits = modelo(imagenes)
            valor = perdida(logits.float(), etiquetas)   # la pérdida siempre en float32 (más estable)
            optimizador.zero_grad(set_to_none=True)
            escalador.scale(valor).backward()
            escalador.step(optimizador)
            escalador.update()
            suma_perdida += valor.item(); n_lotes += 1
        planificador.step()

        acumulador, perdida_valid = evaluar(modelo, cargador_validacion, dispositivo, perdida, configuracion.umbral)
        m = acumulador.metricas()
        fila = {"epoca": epoca + 1, "tasa": optimizador.param_groups[0]["lr"],
                "perdida_train": suma_perdida / max(n_lotes, 1), "perdida_valid": perdida_valid,
                **{f"{k}_valid": v for k, v in m.items()}, "segundos": time.time() - inicio}
        historial.append(fila)

        mejoro = m["iou"] > mejor_iou
        if mejoro:
            mejor_iou, epocas_sin_mejora = m["iou"], 0
            torch.save({"modelo": modelo.state_dict(), "epoca": epoca + 1, "iou_valid": mejor_iou,
                        "configuracion": configuracion.__dict__}, ruta_mejor)
        else:
            epocas_sin_mejora += 1
        torch.save({"modelo": modelo.state_dict(), "optimizador": optimizador.state_dict(),
                    "planificador": planificador.state_dict(), "escalador": escalador.state_dict(),
                    "epoca": epoca, "mejor_iou": mejor_iou, "epocas_sin_mejora": epocas_sin_mejora,
                    "historial": historial}, ruta_ultimo)
        pd.DataFrame(historial).to_csv(dir_salida / "historial.csv", index=False)

        print(f"época {epoca + 1:3d} | pérdida train {fila['perdida_train']:.4f} valid {perdida_valid:.4f} | "
              f"IoU valid {m['iou']:.4f} | F1 {m['f1']:.4f} | {fila['segundos']:.0f}s" + ("  ★" if mejoro else ""))
        if epocas_sin_mejora >= configuracion.paciencia:
            print(f"Parada temprana: {configuracion.paciencia} épocas sin mejorar.")
            break

    resumen = {"nombre": configuracion.nombre, "mejor_iou_valid": mejor_iou,
               "mejor_epoca": int(pd.DataFrame(historial)["iou_valid"].idxmax()) + 1,
               "epocas_entrenadas": len(historial)}
    with open(dir_salida / "resumen.json", "w") as f:
        json.dump(resumen, f, indent=2)
    return pd.DataFrame(historial), resumen


def cargar_mejor_modelo(dir_salida, dispositivo):
    from .experimento import ConfiguracionExperimento
    configuracion = ConfiguracionExperimento.cargar(dir_salida / "configuracion.json")
    modelo = crear_modelo(configuracion).to(dispositivo)
    estado = torch.load(dir_salida / "mejor.pt", map_location=dispositivo, weights_only=False)
    modelo.load_state_dict(estado["modelo"])
    modelo.eval()
    return modelo, configuracion, estado["epoca"]


Writing /content/drive/MyDrive/rnap_p2_inundaciones/src/entrenamiento.py


In [5]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/visualizacion_entrenamiento.py
"""
visualizacion_entrenamiento.py
Curvas de entrenamiento y predicciones de ejemplo.
"""
import matplotlib.pyplot as plt
import numpy as np

from .visualizacion import mostrar_mascara


def figura_curvas(historial, titulo="", linea_base_iou=None, ruta_guardado=None):
    figura, ejes = plt.subplots(1, 3, figsize=(15, 3.8))
    ejes[0].plot(historial["epoca"], historial["perdida_train"], label="train")
    ejes[0].plot(historial["epoca"], historial["perdida_valid"], label="valid")
    ejes[0].set_title("Pérdida"); ejes[0].legend()
    ejes[1].plot(historial["epoca"], historial["iou_valid"], label="IoU", color="#1f5fbf")
    ejes[1].plot(historial["epoca"], historial["f1_valid"], label="F1", color="#7a9cc6")
    if linea_base_iou is not None:
        ejes[1].axhline(linea_base_iou, ls="--", color="gray", label=f"Otsu IoU = {linea_base_iou:.3f}")
    mejor = historial["iou_valid"].idxmax()
    ejes[1].scatter(historial["epoca"][mejor], historial["iou_valid"][mejor], color="red", zorder=5,
                    label=f"mejor: {historial['iou_valid'][mejor]:.3f} (ép. {historial['epoca'][mejor]})")
    ejes[1].set_title("Validación"); ejes[1].legend(fontsize=8)
    ejes[2].plot(historial["epoca"], historial["tasa"], color="black")
    ejes[2].set_title("Tasa de aprendizaje")
    for eje in ejes:
        eje.set_xlabel("época"); eje.grid(alpha=0.3)
    figura.suptitle(titulo)
    figura.tight_layout()
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


def figura_predicciones(ejemplos, umbral=0.5, ruta_guardado=None):
    """ejemplos: [(id_chip, vh_normalizado, etiqueta, probabilidad), ...]"""
    figura, ejes = plt.subplots(len(ejemplos), 4, figsize=(12, 3.1 * len(ejemplos)))
    ejes = np.atleast_2d(ejes)
    for fila, (id_chip, vh, etiqueta, probabilidad) in enumerate(ejemplos):
        ejes[fila, 0].imshow(vh, cmap="gray", vmin=-2.5, vmax=2.5); ejes[fila, 0].set_title("VH normalizado", fontsize=9)
        mostrar_mascara(ejes[fila, 1], etiqueta, "Etiqueta dorada")
        ejes[fila, 2].imshow(probabilidad, cmap="Blues", vmin=0, vmax=1); ejes[fila, 2].set_title("P(agua)", fontsize=9)
        prediccion = np.where(etiqueta == -1, -1, (probabilidad > umbral).astype(int))
        mostrar_mascara(ejes[fila, 3], prediccion, f"Predicción (>{umbral})")
        for eje in ejes[fila, :3]:
            eje.axis("off")
        ejes[fila, 0].text(-0.08, 0.5, id_chip, transform=ejes[fila, 0].transAxes, rotation=90,
                           va="center", ha="right", fontsize=9)
    figura.tight_layout()
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


Writing /content/drive/MyDrive/rnap_p2_inundaciones/src/visualizacion_entrenamiento.py


In [6]:
import pandas as pd
import torch

from src.configuracion import *
from src.utilidades import fijar_semilla, obtener_dispositivo
from src.descarga import descargar_mano
from src.preprocesamiento import Normalizador
from src.cargadores import crear_cargadores
from src.experimento import ConfiguracionExperimento
from src.entrenamiento import entrenar, cargar_mejor_modelo
from src.evaluacion import evaluar, predecir_probabilidades
from src.visualizacion_entrenamiento import figura_curvas, figura_predicciones

dispositivo = obtener_dispositivo()
assert dispositivo.type == 'cuda', 'Esta fase necesita GPU: Entorno de ejecución > Cambiar tipo > T4 GPU'
fijar_semilla(SEMILLA)
descargar_mano()
normalizador = Normalizador.desde_json(DIR_RESULTADOS / 'f1_estadisticas_normalizacion.json')
cargadores = crear_cargadores(normalizador, tamano_lote=8, tamano_recorte=256, num_workers=2, semilla=SEMILLA)
linea_base_otsu = pd.read_csv(DIR_RESULTADOS / 'f4_linea_base_otsu.csv', index_col=0)

OJO: sin GPU. En Colab: Entorno de ejecución > Cambiar tipo > T4 GPU


AssertionError: Esta fase necesita GPU: Entorno de ejecución > Cambiar tipo > T4 GPU

In [7]:
######################### Con CPU demora horas configurar en sección 2 #################################
import os
import pandas as pd
import torch

from src.configuracion import *
from src.utilidades import fijar_semilla, obtener_dispositivo
from src.descarga import descargar_mano
from src.preprocesamiento import Normalizador
from src.cargadores import crear_cargadores
from src.experimento import ConfiguracionExperimento
from src.entrenamiento import entrenar, cargar_mejor_modelo
from src.evaluacion import evaluar, predecir_probabilidades
from src.visualizacion_entrenamiento import figura_curvas, figura_predicciones

dispositivo = obtener_dispositivo()
en_cpu = dispositivo.type == 'cpu'
if en_cpu:
    torch.set_num_threads(os.cpu_count())   # todos los núcleos para las convoluciones
    print(f"Entrenando en CPU con {os.cpu_count()} hilos")

fijar_semilla(SEMILLA)
descargar_mano()
normalizador = Normalizador.desde_json(DIR_RESULTADOS / 'f1_estadisticas_normalizacion.json')
# en CPU, num_workers=0: los workers competirían por los mismos núcleos (los chips ya están en RAM)
cargadores = crear_cargadores(normalizador, tamano_lote=8, tamano_recorte=256,
                              num_workers=0 if en_cpu else 2, semilla=SEMILLA)
linea_base_otsu = pd.read_csv(DIR_RESULTADOS / 'f4_linea_base_otsu.csv', index_col=0)


OJO: sin GPU. En Colab: Entorno de ejecución > Cambiar tipo > T4 GPU
Entrenando en CPU con 2 hilos
[descargando] S1Hand ...
[descargando] LabelHand ...
[descargando] S1OtsuLabelHand ...
[descargando] flood_handlabeled ...


Cargando chips en RAM:   0%|          | 0/252 [00:00<?, ?it/s]

train   : 252 chips ->  31 lotes


Cargando chips en RAM:   0%|          | 0/89 [00:00<?, ?it/s]

valid   :  89 chips ->  23 lotes


Cargando chips en RAM:   0%|          | 0/90 [00:00<?, ?it/s]

test    :  90 chips ->  23 lotes


Cargando chips en RAM:   0%|          | 0/15 [00:00<?, ?it/s]

bolivia :  15 chips ->   4 lotes


## 2. Configuración del experimento base

In [ ]:
configuracion = ConfiguracionExperimento(nombre='base_bce', perdida='bce')
DIR_EXPERIMENTO = DIR_RESULTADOS / 'experimentos' / configuracion.nombre
configuracion

In [8]:
######################### Con CPU demora horas configurar en sección 2 #################################
configuracion = ConfiguracionExperimento(nombre='base_bce', perdida='bce', epocas=60, paciencia=15)
DIR_EXPERIMENTO = DIR_RESULTADOS / 'experimentos' / configuracion.nombre
configuracion

ConfiguracionExperimento(nombre='base_bce', fuente_entrenamiento='mano', tamano_lote=8, tamano_recorte=256, canales_base=32, profundidad=4, usar_saltos=True, abandono=0.0, perdida='bce', parametros_perdida={}, tasa_aprendizaje=0.001, decaimiento_pesos=0.0001, epocas=60, epocas_calentamiento=3, paciencia=15, umbral=0.5, semilla=42)

## 3. Entrenar
Imprime una línea por época; ★ marca cuando mejora el IoU de validación (y se guarda `mejor.pt`).
**Si Colab se desconecta:** vuelve a correr desde la sección 0 hasta aquí → reanuda automáticamente.

In [ ]:
historial, resumen = entrenar(configuracion, cargadores['train'], cargadores['valid'], dispositivo, DIR_EXPERIMENTO)
resumen

época   1 | pérdida train 0.3913 valid 0.3234 | IoU valid 0.5416 | F1 0.7026 | 611s  ★
época   2 | pérdida train 0.2300 valid 0.1756 | IoU valid 0.5466 | F1 0.7068 | 605s  ★
época   3 | pérdida train 0.1938 valid 0.1630 | IoU valid 0.5417 | F1 0.7027 | 605s
época   4 | pérdida train 0.1898 valid 0.1507 | IoU valid 0.5742 | F1 0.7295 | 590s  ★
época   5 | pérdida train 0.1703 valid 0.1646 | IoU valid 0.5683 | F1 0.7247 | 585s
época   6 | pérdida train 0.1810 valid 0.1428 | IoU valid 0.5879 | F1 0.7404 | 588s  ★
época   7 | pérdida train 0.1806 valid 0.1552 | IoU valid 0.5807 | F1 0.7347 | 589s
época   8 | pérdida train 0.1837 valid 0.1548 | IoU valid 0.5841 | F1 0.7375 | 592s
época   9 | pérdida train 0.1669 valid 0.1471 | IoU valid 0.5475 | F1 0.7076 | 589s
época  10 | pérdida train 0.1502 valid 0.1540 | IoU valid 0.4869 | F1 0.6550 | 592s
época  11 | pérdida train 0.1596 valid 0.1799 | IoU valid 0.5544 | F1 0.7133 | 589s
época  12 | pérdida train 0.1545 valid 0.1338 | IoU valid 0.5726

## 4. Curvas de entrenamiento
**Qué observar:**
- Pérdida train y valid deben bajar. Si train sigue bajando y valid sube → **sobreajuste**.
- El IoU de valid debería superar la línea punteada de Otsu (0.507 en valid).

In [ ]:
figura_curvas(historial, 'U-Net base (BCE, solo dorados)', linea_base_otsu.loc['valid', 'iou'],
              DIR_FIGURAS / 'f5_curvas_base_bce.png');

## 5. Mejor modelo vs. Otsu en validación
(Recuerda: **test** se evalúa solo en la fase 7.)

In [ ]:
modelo, _, mejor_epoca = cargar_mejor_modelo(DIR_EXPERIMENTO, dispositivo)
acumulador_valid, _ = evaluar(modelo, cargadores['valid'], dispositivo)
comparacion = pd.DataFrame({'Otsu': linea_base_otsu.loc['valid'], f'U-Net base (ép. {mejor_epoca})': acumulador_valid.metricas()}).T
display(comparacion[['iou', 'f1', 'precision', 'recall', 'omision', 'comision']].round(3))

por_evento = pd.concat({'U-Net': acumulador_valid.tabla_por_evento()['iou']}, axis=1).round(3)
por_evento

## 6. Predicciones de ejemplo (validación)

In [ ]:
conjunto_valid = cargadores['valid'].dataset
eventos_vistos, ejemplos = set(), []
for i, id_chip in enumerate(conjunto_valid.ids_chips):
    evento = id_chip.split('_')[0]
    _, etiqueta = conjunto_valid.cache[i]
    if evento in eventos_vistos or (etiqueta == 1).mean() < 0.05:
        continue
    eventos_vistos.add(evento)
    imagen, etiqueta, _ = conjunto_valid[i]
    ejemplos.append((id_chip, imagen[1].numpy(), etiqueta.numpy(),
                     predecir_probabilidades(modelo, imagen, dispositivo).numpy()))
    if len(ejemplos) == 4:
        break
figura_predicciones(ejemplos, ruta_guardado=DIR_FIGURAS / 'f5_predicciones_base_bce.png');

## Para traer a la conversación
El `resumen` (sección 3), las curvas (4), la tabla comparativa y por evento (5) y la figura de predicciones (6).
Con eso decidimos los experimentos de la ablación.